# Lab 1 — Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 1 — The task, and preparing the data

### The task in one sentence

Given a non-disclosure agreement, answer a **fixed 17-point legal checklist** about
it, and for each answer **cite the clause numbers** that justify it.

### The input

One request contains the instruction, the contract split into numbered spans, and
the checklist — in that order. This is the real prompt template, with the contract
truncated and the checklist showing 3 of its 17 items:

```
You are a contract review assistant. You review a non-disclosure agreement (NDA)
against a fixed checklist of 17 legal hypotheses.

For EACH hypothesis, decide:
- "Entailment": the contract states or implies the hypothesis is true.
- "Contradiction": the contract states something that conflicts with the hypothesis.
- "NotMentioned": the contract does not address it.

Also cite the span numbers that justify the decision (the exact spans a lawyer
would point to). Cite spans only for Entailment or Contradiction; use an empty
list for NotMentioned. Read exceptions and carve-outs carefully: a clause with an
exception may contradict a hypothesis stated absolutely.

CONTRACT (numbered spans):
[0] NAVIDEC, INCORPORATED
[1] TRADE SECRET/NON-DISCLOSURE AGREEMENT
[2] In consideration of the mutual promises made herein, as well as the agreement
    between Navidec, Incorporated and _______ , the parties hereby agree as follows:
[3] _______ , agrees that, in consideration for being shown or told about certain
    trade secrets or property belonging to Navidec, Incorporated, _______ , shall
    not disclose or cause to be disclosed, disseminated or distributed any
    information concerning said trade secret or property to any person, entity,
    business or other individual or company without the prior written permission
    of Navidec, Incorporated.
[4] Further, _______ , agrees not to use, either directly or indirectly any of the
    material, ideas, objects or portions thereof of said trade secret or property
    disclosed by Navidec, Incorporated in any manner whatsoever without the prior
    written consent of Navidec, Incorporated.
[5] Any dispute that arises hereunder shall be resolved by arbitration pursuant to
    the rules of the American Arbitration Association or the rules of the State of
    Colorado.
    ... (remaining spans)

CHECKLIST:
nda-2: Confidential Information shall only include technical information.
       (None-inclusion of non-technical information)
nda-7: Receiving Party may share some Confidential Information with some
       third-parties (including consultants, agents and professional advisors).
       (Sharing with third-parties)
nda-5: Receiving Party may share some Confidential Information with some of
       Receiving Party's employees. (Sharing with employees)
    ... (17 items total, always in the same order)

Respond with JSON only, no other text:
{"nda-1": {"label": "Entailment|Contradiction|NotMentioned", "evidence": [span numbers]}, ...}
Include an entry for every hypothesis key listed above.
```

The instruction and the checklist are sent as a **system** turn, the contract as a
**user** turn, and the JSON above as the **assistant** turn. `contractnli.py` builds all
three in `build_messages()`, and the model's own chat template turns them into the string
the model actually reads — at training time and at serving time, from the same function.

**One flag has to travel with every request.** The base model is a reasoning model: asked
to deliberate over 17 hypotheses it produces thousands of tokens of `<think>` and runs out
of generation budget before it ever emits the JSON. The switch that turns that off is not
a string in the prompt — on some reasoning models `/no_think` does it, but **this model's
chat template ignores it** and takes an explicit `enable_thinking` flag instead. It lives
once in `contractnli.py` as `C.CHAT_TEMPLATE_KWARGS`, and the section on building records
below shows the two places it is passed.

Note the carve-out sentence in the instruction: **carve-outs matter**. That single
sentence is what the example below turns on.

Everything except the contract — instruction, checklist, required output shape — is a
fixed 3,234 characters, identical in every record. The whole contract goes in every time,
because any of the 17 items could be decided by any clause. You will measure the real
token lengths with this model's own tokeniser further down, since that is what decides
`max_length` in the training recipe.

### The expected output

The **completion**, and it is strict JSON. One key per checklist item, in the checklist's order, each with a
verdict and a list of span numbers. This is the complete, exact answer for the
contract above — all 17 items, nothing omitted:

```json
{"nda-11": {"label": "NotMentioned",  "evidence": []},
 "nda-16": {"label": "NotMentioned",  "evidence": []},
 "nda-15": {"label": "NotMentioned",  "evidence": []},
 "nda-10": {"label": "NotMentioned",  "evidence": []},
 "nda-2":  {"label": "Contradiction", "evidence": [3, 4]},
 "nda-1":  {"label": "NotMentioned",  "evidence": []},
 "nda-19": {"label": "NotMentioned",  "evidence": []},
 "nda-12": {"label": "NotMentioned",  "evidence": []},
 "nda-20": {"label": "NotMentioned",  "evidence": []},
 "nda-3":  {"label": "NotMentioned",  "evidence": []},
 "nda-18": {"label": "NotMentioned",  "evidence": []},
 "nda-7":  {"label": "Contradiction", "evidence": [3]},
 "nda-17": {"label": "NotMentioned",  "evidence": []},
 "nda-8":  {"label": "NotMentioned",  "evidence": []},
 "nda-13": {"label": "NotMentioned",  "evidence": []},
 "nda-5":  {"label": "Contradiction", "evidence": [3]},
 "nda-4":  {"label": "Entailment",    "evidence": [4]}}
```

### Why those are the right answers

Take `nda-5`: *"Receiving Party may share some Confidential Information with some
of Receiving Party's employees."*

Span [3] forbids disclosure **"to any person, entity, business or other individual
or company"** with no exception for employees. Most NDAs carve one out; this one
does not. So the checklist statement is not merely unaddressed — it is
**contradicted**, and span [3] is the proof.

`nda-7` (sharing with third parties) is contradicted by the same span. `nda-2`
(*"Confidential Information shall only include technical information"*) is
contradicted by [3] and [4] together, which cover "material, ideas, objects" and
"trade secrets or property" far beyond technical information. And `nda-4`
(*"shall not use Confidential Information for any purpose other than the purposes
stated in the Agreement"*) is **entailed** by [4].

Two things to notice, because they define the difficulty:

- **One clause drives several answers.** Span [3] decides three items. This is not
  retrieval where each question has its own passage.
- **13 of 17 items are `NotMentioned` in this contract.** Short NDAs are silent on
  most of the checklist. Across the whole test split the skew is milder — 43%
  `NotMentioned`, 46% `Entailment`, 10% `Contradiction` — so a model that ignores the
  contract and always answers `NotMentioned` still scores 43% accuracy for free. That
  is the floor any accuracy number has to be read against, and notebook 3 does.

### Why the citations are the point

A verdict alone is unusable. "Item 5 fails" gives a reviewer nothing to act on;
"Item 5 fails, see clause 3" is verifiable in seconds. The citation is what turns
model output into reviewable work, and it is also the part that cannot be faked —
guessing the most common verdict gets you 43% accuracy but scores **exactly zero**
on evidence.

### The same pattern shows up far beyond legal work

The shape of this task, not its subject, is what makes it worth studying: **a fixed
checklist, applied repeatedly to documents of one type, where the output is a
verdict plus a pointer to the text that justifies it.** Swap the checklist and the
document and you have vendor security review, insurance claims adjudication,
regulatory filing checks, clinical trial screening, RFP compliance. All of them
share the two failure modes that matter here: claiming a requirement is satisfied
when the document is actually silent, and pointing at the wrong evidence. This lab
is about measuring and fixing exactly those.

### Is fine-tuning the right tool here?

Yes, and it is measured rather than assumed. On the full 123-contract held-out set
(2,091 decisions), the base model scores 64.8 accuracy / 48.9 evidence-F1, Claude
Sonnet 5 zero-shot scores 83.6 / 67.1, and the model you are about to train reaches
**83.8 / 68.9** — parity with the frontier model, at roughly 8x lower cost per
contract. Read that as parity rather than a win: the margin is a fraction of a point
on accuracy, and the two rows are not averaged the same way (notebook 3 explains
which). The gain that matters is the one over the starting point — **+19 accuracy and
+20 evidence-F1** over the same base weights, on a task the base model was already
answering in valid JSON. Putting worked examples in the prompt makes the small model
*worse*, not better, which is the clearest sign this is a fine-tuning problem rather
than a prompt-engineering one.

Those figures come from one run using the settings notebook 2 configures. A
fine-tune is not bit-reproducible and 315 training records is few, so expect several
points of movement rather than these decimals.

The full results table, the cost breakdown with its break-even point, and the
few-shot experiment are in the workshop guide (Lab 1 overview and Evaluation
pages). Notebooks 2 and 3 are where you reproduce the numbers.

### The data

[ContractNLI](https://stanfordnlp.github.io/contract-nli/) (Koreeda & Manning,
*Findings of EMNLP 2021*), released under **CC-BY-4.0**. 607 real NDAs from EDGAR
filings and the public web, annotated against this fixed 17-point checklist.

For every (contract, checklist item) pair the annotation gives:

| Field | Meaning |
|---|---|
| `choice` | `Entailment` / `Contradiction` / `NotMentioned` |
| `spans` | the span numbers a lawyer would point to as justification |

Splits are **document-level** and stratified by source format, so no contract
appears in two splits: 423 train / 61 dev / 123 test.


### Install requirements

In [ ]:
%pip install -r requirements.txt

#### Setup and dependencies

Standard SageMaker boilerplate, no task-specific logic: it resolves the execution role,
the default bucket and the bucket prefix (`None` unless a SageMaker defaults config sets
one), and opens the boto3 clients. Of all that, this notebook reuses only `s3_client`,
`bucket_name` and `default_prefix`, all three in the upload cell; the role is what
notebook 2 passes to the training job. Notebooks 2, 3 and 4 repeat this cell verbatim.

The part worth reading is the `try`/`except`. `get_execution_role()` returns the caller
identity when it is already a role — the SageMaker Studio case — and raises `ValueError`
when it is not, for example when you run locally as an IAM user. The fallback looks up a
role named literally `sagemaker_execution_role`, so change that name if yours differs.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

### First: what is `contractnli.py`?

Every notebook in this lab starts with `import contractnli as C`. It is a small module
that sits next to these notebooks, and it holds only the unglamorous parts — downloading
the data, reading it, and rendering the prompt. Anything that constitutes the *lesson*
(calling a model, scoring its answer) stays in the notebooks where you can see it.

You will see these names used throughout. This is the whole surface:

| Call | Returns | Used for |
|---|---|---|
| `C.ensure_dataset("./data")` | the unpacked path | downloads the ContractNLI archive once |
| `C.load(split)` | `(documents, checklist)` | reads `train`, `dev` or `test` |
| `C.doc_spans(doc)` | `[(0, "text"), (1, "text"), ...]` | one contract as numbered clauses |
| `C.gold_for(doc)` | `{"nda-1": {"choice", "spans"}, ...}` | the expert answer for one contract |
| `C.build_messages(doc, labels)` | `[system, user]` turns | **the request every caller sends** |
| `C.build_messages(doc, labels, completion=...)` | `[system, user, assistant]` | a full training record |

Two constants matter too. `C.INSTRUCTION` is the template the system turn is built from,
and `C.CHAT_TEMPLATE_KWARGS` is `{"enable_thinking": False}` — the flag that stops the
model reasoning past its budget. You will print both in a moment.

**`build_messages` is the only rendering that trains or serves anything.** The same two
turns become the `prompt` of a training record here, the request sent to the deployed
endpoint in notebook 3's smoke test, the requests notebook 4 scores, and the turns the
Bedrock baseline is given. One function, so they cannot drift.

`C.build_prompt(doc, labels)` renders the same content as one flat string. Nothing trains
or serves through it — it exists so this notebook can print a whole request and measure it
in one piece.

Open the file if you want — it is about 150 lines and deliberately boring. The cells
below show what each helper returns rather than asking you to take it on trust.


### Download the dataset

`ensure_dataset()` fetches the CC-BY-4.0 archive from Stanford and unpacks it
into `./data`. Nothing else in the lab needs network access to the dataset.

`C.load(split)` then reads that split's JSON and returns `(documents, checklist)`.
Each document carries its text, the character offsets that cut it into numbered spans,
and the expert annotation. The checklist is byte-identical in all three files — same 17
items, same order — so only the train copy is kept, as `labels`, and the dev and test
copies go to `_`. That one dict renders the checklist into every prompt, in all three
splits.

In [ ]:
import contractnli as C

C.ensure_dataset("./data")

train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")

print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

#### What one document actually looks like

Before building anything, it is worth seeing the raw shape you are working from. The
dataset gives you documents; the three helpers below are how you get from a document to
the pieces the prompt needs.


In [ ]:
doc_example = train_docs[0]

print("One ContractNLI document is a plain dict. Its fields:\n")
for key, value in doc_example.items():
    size = f"{len(value):,} items" if isinstance(value, list) else f"{len(str(value)):,} chars"
    print(f"  doc[{key!r}]:{' ' * (20 - len(key))}{type(value).__name__:5s} {size}")

print("\nOnly three of those matter here, and `contractnli.py` has a helper for each.\n")

# 1. The contract, cut into the clauses the model will cite by number.
print("1. doc['spans'] holds (start, end) offsets into doc['text'] — the dataset's own")
print("   clause split. C.doc_spans(doc) slices them out and numbers them:\n")
for number, text in C.doc_spans(doc_example)[:3]:
    print(f"     [{number}] {text[:62]}")

# 2. The expert answer. `choice` is the verdict, `spans` the clauses that justify it.
print("\n2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it")
print("   to one entry per checklist item:\n")
gold_example = C.gold_for(doc_example)
for key in list(labels)[:2]:
    print(f"     {key}: {gold_example[key]}")

# 3. The checklist is the same for every contract. In this format it is rendered into
#    every prompt, which is why it comes from one dict rather than per-record text.
print("\n3. `labels`, the second value C.load() returned, is the checklist itself:\n")
first_key = list(labels)[0]
print(f"     labels[{first_key!r}]:")
for field, text in labels[first_key].items():
    print(f"       {field}: {text[:66]}")

### The checklist the model has to answer

The 17 hypotheses, each with the short description that names it. Not background
reading: `build_system()` renders these same two fields, one line per item, into the
`CHECKLIST:` block of the system turn — 2,324 characters of the 3,234 every request
carries besides the contract itself. `labels` is
loaded once, from the train file, and the same dict builds all three splits, so every
contract is judged against this list.

The `int(...)` sort key is for readability only: the dataset's order starts at
`nda-11`, and plain sorting would put `nda-10` before `nda-2`. Numbers 6, 9 and 14 are
unused, which is why 17 items reach `nda-20`. The prompt and the completion both
keep the dataset's order — that is why the JSON in the introduction starts at `nda-11`.

In [ ]:
for k, v in sorted(labels.items(), key=lambda kv: int(kv[0].split("-")[1])):
    print(f"{k:7s} [{v['short_description']}]")
    print(f"        {v['hypothesis']}")

### Look at one real contract

Below is one of the shortest NDAs in the test set, split into the numbered spans
the model will see, followed by the gold answer. Read span [3] and then look at
the labels for `nda-5` (sharing with employees) and `nda-7` (sharing with
third-parties).

In [ ]:
doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]
spans = C.doc_spans(doc)

print(f"{doc['file_name']}  —  {len(doc['text'].split())} words, {len(spans)} spans\n")
for i, t in spans:
    print(f"[{i}] {t[:160]}")

The gold answer for the same contract, one line per checklist item. `C.gold_for()`
returns the `annotations` of the document's single annotation set: per item, a `choice`
from the three labels and `spans` — the span numbers the annotator pointed to, in the
same numbering as the `[i]` markers above. `spans` is non-empty for exactly the
`Entailment` and `Contradiction` entries, in all 10,319 judgements across the three
splits, so the citation rule the prompt states is one the data already obeys.

`gold_json()` further down turns this into the completion, renaming `choice` to
`label` and `spans` to `evidence`. It emits the items in the checklist's order rather
than the numeric order sorted here, which is why the JSON in the introduction starts at
`nda-11`.

In [ ]:
gold = C.gold_for(doc)
print("gold answer:\n")
for k in sorted(gold, key=lambda x: int(x.split("-")[1])):
    v = gold[k]
    ev = f"  evidence={v['spans']}" if v["spans"] else ""
    print(f"{k:7s} {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

### Check the answer against the clause you just read

This is the contract from the introduction, and span [3] is that blanket
prohibition. Notice it drives three separate `Contradiction` verdicts:

| Item | Subject | Evidence |
|---|---|---|
| `nda-5` | sharing with employees | `[3]` |
| `nda-7` | sharing with third parties | `[3]` |
| `nda-2` | only technical information is confidential | `[3, 4]` |

One clause, three verdicts — which is why the model has to reason over the whole
document per item rather than retrieve one passage per question.

Notice also that 13 of the 17 items are `NotMentioned`. Short NDAs are silent on most of
the checklist, and that skew is why accuracy alone is a weak metric here: over the whole
test split 43% of decisions are `NotMentioned`, so always answering it scores 43% without
reading anything — see notebook 3.


### The prompt

This is the single most important design decision in the lab, so read it rather than
skim it. Everything the model will ever see is one string: the template below with three
slots filled in — the number of checklist items, the contract as numbered spans, and the
checklist itself.

**The format: chat completion.** Serverless customization accepts a record as a
`prompt`/`completion` pair, which is the shape this notebook writes. Everything the
model reads goes in `prompt`, and the single thing it must produce goes in `completion`;
there are no roles and no turn boundaries to get right. The order inside the prompt is
instruction, then contract, then checklist, then the required output shape — the
checklist sits *after* the contract so the last thing the model reads before answering
is what it is being asked.

It is defined **once**, in `contractnli.py`, and used by every notebook — data prep
here, the endpoint smoke test in notebook 3, and both the scored requests and the frontier
baseline in notebook 4. That is deliberate: the promise that *the training request is the
inference request* only holds if there is exactly one copy. A second copy pasted into a
notebook is how train/serve skew gets introduced.

Print it, change it, experiment — but change it in one place.


In [ ]:
# The exact template. {n}, {spans} and {checklist} are the only substitutions.
print("=" * 70, "\nINSTRUCTION\n", "=" * 70, sep="")
print(C.INSTRUCTION)

# Reasoning is turned off by a flag on the chat template, not by a string in the prompt.
# This dict is what travels with every record and every request.
print(f"\nC.CHAT_TEMPLATE_KWARGS = {C.CHAT_TEMPLATE_KWARGS}")
print("  -> the template renders `<|im_start|>assistant\\n<think></think>` instead of")
print("     opening a <think> block, so the model answers directly.")


To experiment with the wording, set `C.INSTRUCTION` here and re-run the record build
below — every notebook then picks up your version:

```python
C.INSTRUCTION = """...your wording, keeping {n}, {spans} and {checklist}..."""
```

The assertions further down are what protect the no-skew promise: the turns stored in
each training record must be exactly what `C.build_messages(doc, labels)` produces at
inference time, and every record must carry `enable_thinking=False`.


### Build the training records

One training record = one contract, with all 17 verdicts in the completion. So
423 records — but each carries 17 supervised decisions plus the evidence spans,
which is roughly 7,200 labelled judgements.

**The format: conversational `prompt`/`completion`.** Two columns, each a list of chat
turns rather than a string:

```json
{"prompt": [{"role": "system",    "content": "You are a contract review assistant... CHECKLIST: ..."},
            {"role": "user",      "content": "CONTRACT (numbered spans):\n[0] ..."}],
 "completion": [{"role": "assistant", "content": "{\"nda-11\": {\"label\": \"NotMentioned\", ...}}"}],
 "chat_template_kwargs": {"enable_thinking": false}}
```

Three things this shape buys, and they are the reason it is not a single string:

- **Only the completion is supervised.** TRL sees a `prompt` and a `completion` column and
  turns on completion-only loss by itself: it renders the template over `prompt` alone and
  over `prompt + completion`, and masks the token-length difference. So the model is
  trained to *produce* the JSON, not to reproduce the contract.
- **The model's own chat template does the rendering, inside the job.** We never write
  `<|im_start|>` by hand. That matters because the same template runs in the vLLM
  container at serving time, so there is no hand-rolled string to drift.
- **`chat_template_kwargs` rides along per record.** TRL forwards it to
  `apply_chat_template`, which is how reasoning gets switched off during training — see
  below.

#### Turning reasoning off is a flag, not a magic string

Nemotron 3 Nano is a reasoning model. Asked to deliberate over 17 hypotheses it will
spend its whole generation budget inside `<think>` and get cut off before the JSON. On
some reasoning models you suppress that with a `/no_think` string in the prompt. **Not
this one** — its chat template ignores unknown text and takes an explicit flag:

```jinja
{%- if enable_thinking %}
    {{- '<|im_start|>assistant\n<think>\n' }}
{%- else %}
    {{- '<|im_start|>assistant\n<think></think>' }}
{%- endif %}
```

With the flag false the template pre-fills an empty think block, so there is no open
`<think>` for the model to continue and it answers directly. `contractnli.py` holds the
dict once, as `C.CHAT_TEMPLATE_KWARGS`, and it reaches the template two ways: as this
column at training time, and as `chat_template_kwargs` in the request body at serving
time. Same flag, one definition.

#### One thing to check before you train: sequence length

`args.yaml` sets `max_length: 8192`, and records over it are **truncated** — they still
train, minus whatever fell outside the window. The cell below measures the real
distribution with this model's own tokeniser over the full chat template, which is the
only count that matters, so you can see what that cap actually costs on this dataset
rather than trusting the default.

The alternative — one record per (contract, checklist item), 7,191 short records — gives
better gradient diversity but re-encodes the whole contract 17 times, so roughly 17x the
training compute. Not worth it here.

#### All three splits get the same shape

Train and validation go to S3 as Training-job channels. The **test** split gets the same
`prompt`/`completion` columns and stays on local disk: notebook 4 reads it, sends each
`prompt` to the deployed endpoint, and scores the reply against the `completion`. There
is no second format to keep in sync, and nothing registers the test set anywhere.


#### The three pieces, one contract at a time

The next cell builds all 607 records at once, which makes it hard to see what any single
line does. So here is the same work on one contract, step by step. Nothing below is
needed later — it is here to be read.


In [ ]:
import json

label_keys = list(labels.keys())          # the checklist's own order, reused below


def gold_json(doc):
    """The expert answer for one contract, as the exact JSON the model must emit."""
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})


# Step 1 — the target. gold_json only renames fields; the judgement is the annotator's.
item = label_keys[1]
print("the dataset stores:      ", json.dumps(C.gold_for(doc_example)[item]))
print("the prompt asks for:     ", json.dumps(json.loads(gold_json(doc_example))[item]))
print("                          ^ choice -> label, spans -> evidence, for all 17 items")

# Step 2 — the turns. build_messages returns system + user; pass the completion and it
# appends the assistant turn too. The record splits them: prompt = the first two,
# completion = the last one.
target = gold_json(doc_example)
turns = C.build_messages(doc_example, labels, completion=target)

print(f"\nC.build_messages(doc, labels, completion=...) -> {len(turns)} turns")
for turn in turns:
    print(f"  {turn['role']:9s} {len(turn['content']):6,d} chars")

# Step 3 — the record. Two list-valued columns plus the flag that turns reasoning off.
record = {"prompt": turns[:-1],
          "completion": turns[-1:],
          "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS}
print(f"\nrecord keys: {list(record)}")
print("\nOne function, every caller: the same build_messages() turns are the `prompt` of a")
print("training record here, the request sent to the endpoint in notebook 4, and the")
print("Bedrock baseline's turns. That is why they cannot drift apart.")


Now all three splits at once. Three things happen, in order.

1. **`make_records`** maps documents to records — the same builder for all three splits,
   since they now share one shape. `build_messages` produces the turns, the last one is
   split off as the `completion`, and `C.CHAT_TEMPLATE_KWARGS` is attached to every row.
2. **The token measurement** renders each record through the model's real chat template
   and counts tokens, so the `max_length: 8192` in `args.yaml` is a decision you can see
   the cost of rather than a default you inherited.
3. **The assertions** are the no-skew promise made executable: every stored `prompt` must
   equal `C.build_messages(doc, labels)` for its document, every completion must parse as
   non-empty JSON, and every row must carry the reasoning-off flag. Rewording
   `C.INSTRUCTION` and rebuilding still passes, since both sides re-render from it; what
   fails is a prompt assembled by any other route.


In [ ]:
# `label_keys` and `gold_json` come from the cell above. One builder now, for all three
# splits: the shape no longer differs between training and evaluation.

def make_records(docs):
    """One record per contract: prompt turns, completion turn, reasoning-off flag."""
    out = []
    for d in docs:
        turns = C.build_messages(d, labels, completion=gold_json(d))
        out.append({"prompt": turns[:-1],
                    "completion": turns[-1:],
                    "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS})
    return out


records = {"train": make_records(train_docs),
           "val": make_records(dev_docs),
           "test": make_records(test_docs)}

for name, rows in records.items():
    print(f"{name:5s}: {len(rows):4d} records")

# The byte-identical promise, enforced rather than asserted in prose.
for split, docs in (("train", train_docs), ("val", dev_docs), ("test", test_docs)):
    for rec, doc in zip(records[split], docs):
        assert set(rec) == {"prompt", "completion", "chat_template_kwargs"}, (
            f"unexpected columns in {split}: {sorted(rec)}")
        assert rec["prompt"] == C.build_messages(doc, labels), (
            f"prompt drift in {split}: stored turns differ from build_messages()")
        assert rec["completion"][0]["role"] == "assistant", f"bad completion in {split}"
        assert json.loads(rec["completion"][0]["content"]), f"empty target in {split}"
        assert rec["chat_template_kwargs"] == {"enable_thinking": False}, (
            f"reasoning not disabled in {split}")

total = sum(len(rows) for rows in records.values())
print(f"\nverified: all {total} stored prompts are identical to build_messages() output,")
print("and every record carries enable_thinking=False")


#### What the cap actually costs

The measurement behind `max_length: 8192`. Each record is rendered through the model's
own chat template — the same template TRL will apply inside the job, with the same
`enable_thinking=False` — and tokenised, so these are the counts the trainer will see.

Only the tokeniser is downloaded here, not the 8 GB of weights.


In [ ]:
from transformers import AutoTokenizer

from config import BASE_MODEL_ID

MAX_LENGTH = 8192          # keep in step with `max_length` in scripts/args.yaml

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

lengths = []
for rec in records["train"]:
    rendered = tokenizer.apply_chat_template(
        rec["prompt"] + rec["completion"],
        tokenize=True,
        # transformers 5 returns a BatchEncoding here, whose len() is its key count
        # rather than a token count. Ask for the plain list of ids instead.
        return_dict=False,
        **rec["chat_template_kwargs"],
    )
    lengths.append(len(rendered))

lengths.sort()
over = sum(n > MAX_LENGTH for n in lengths)
print(f"train records: {len(lengths)}")
print(f"  median  {lengths[len(lengths) // 2]:6,d} tokens")
print(f"  mean    {sum(lengths) // len(lengths):6,d}")
print(f"  p95     {lengths[int(0.95 * len(lengths))]:6,d}")
print(f"  longest {lengths[-1]:6,d}")
print(f"\nover max_length={MAX_LENGTH:,}: {over} of {len(lengths)} records "
      f"({over / len(lengths):.1%}) — truncated, not dropped")

# Confirm the flag reached the template rather than assuming it: with reasoning off the
# rendered assistant turn opens with an already-closed think block.
rendered_text = tokenizer.apply_chat_template(
    records["train"][0]["prompt"] + records["train"][0]["completion"],
    tokenize=False,
    **records["train"][0]["chat_template_kwargs"],
)
assert "<think></think>" in rendered_text, (
    "enable_thinking=False did not reach the chat template — the model will reason "
    "through its whole budget and be cut off before the JSON")
print("\nenable_thinking=False reaches the template (rendered `<think></think>`) \u2713")


#### Write to disk and upload to Amazon S3

A Training job takes its data as **channels**: S3 prefixes that SageMaker copies onto the
instance under `/opt/ml/input/data/<channel>/` before your script runs. So the only thing
notebook 2 needs from this notebook is two S3 prefixes — there is no dataset registry in
this flow, and nothing to look up by name.

That is the one real difference from the serverless customization version of this lab.
There, the splits were registered as AI Registry `DataSet` entries and the trainer
resolved them by name. `ModelTrainer` resolves channels by URI instead, so the splits
stay plain S3 objects and the paths are what get passed forward.

| split | where it goes | read by |
|---|---|---|
| `train` | `s3://<bucket>/[<prefix>/]datasets/contractnli-nda-review/train/dataset.jsonl` | the training job, `train` channel |
| `val` | `.../val/dataset.jsonl` | the training job, `val` channel |
| `test` | `./tmp/test.jsonl`, **local only** | notebook 4, against the deployed endpoint |

The test split is never uploaded. Nothing in the training job reads it, and the
evaluation notebook runs in this same directory — so keeping it on disk means one less
resource to clean up, and the held-out set cannot accidentally be handed to a trainer.

Note the local staging directory is `./sft_data`, not `./data`: `C.ensure_dataset()`
unpacked the ContractNLI archive into `./data/contract-nli`, and this cell removes its
staging directory on every run.


In [ ]:
import pathlib
import shutil

from config import DATA_PREFIX

# ./data holds the downloaded ContractNLI archive — stage the JSONL somewhere else so a
# re-run cannot delete it.
local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)

for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

input_path = f"{default_prefix}/{DATA_PREFIX}" if default_prefix else DATA_PREFIX

# Only train and val become channels. The test split stays local for notebook 4.
train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"

for name in ("train", "val"):
    s3_client.upload_file(str(local / name / "dataset.jsonl"),
                          bucket_name, f"{input_path}/{name}/dataset.jsonl")

test_path = pathlib.Path("./tmp")
test_path.mkdir(parents=True, exist_ok=True)
shutil.copy(local / "test" / "dataset.jsonl", test_path / "test.jsonl")

shutil.rmtree(local)

print("uploaded for the training job:")
print(f"  {train_dataset_s3_path}")
print(f"  {val_dataset_s3_path}")
print(f"\nheld out locally for notebook 4:")
print(f"  {test_path / 'test.jsonl'} ({len(records['test'])} contracts)")


### What you built

Three JSONL files in the conversational `prompt`/`completion` shape, every record
carrying `enable_thinking=False`:

- `train` (423 records) and `val` (61) in S3, ready to mount as Training-job channels
- `test` (123 contracts) on local disk, held back for notebook 4

Nothing else was created — no dataset registry entry, no model package group. Notebook 2
takes the two S3 prefixes above and the recipe in `scripts/args.yaml`, and nothing else
defined in this notebook has to survive.

Continue to **notebook 2** to run the LoRA fine-tuning job on SageMaker Training.
